# Read From Silver Table


### Init


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

In [0]:
def log(step, message):
    print(f"[{step}] {message}")

# Gold Customer 


In [0]:
log("Gold", "Loading Silver data")

df = spark.table("silver.orders")

log("Gold", f"Rows loaded: {df.count()}")
log("Gold", f"Columns available: {df.columns}")

In [0]:
required_cols =  ['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'segment', 'country', 'city', 'execution_datetime', 'source_file', 'customer_first_name', 'customer_last_name']

for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

log("Gold", "Schema validation passed")

## Business Transformation and Modeling


In [0]:
log("Gold-Customer", "Starting customer aggregation")

latest_date = df.agg(F.max("order_date").alias("max_date")).first()["max_date"]

log("Gold-Customer", f"Latest date detected: {latest_date}")

In [0]:
#create flag for which order falls within 1, 6 and 12 months
df_flags = (
    df
    .select(
        "customer_id",
        "customer_first_name",
        "customer_last_name",
        "segment",
        "country",
        "order_date",
        "execution_datetime", 
        "source_file"
    )
    .withColumn("is_last_1m",  F.col("order_date") >= F.date_sub(F.lit(latest_date), 30))
    .withColumn("is_last_6m",  F.col("order_date") >= F.date_sub(F.lit(latest_date), 180))
    .withColumn("is_last_12m", F.col("order_date") >= F.date_sub(F.lit(latest_date), 365))
)

In [0]:
#repartition data by customer_id then aggr to calc order counts for different time periods

df_flags = df_flags.repartition("customer_id")
customer_df = (
    df_flags
    .groupBy(
        "customer_id",
        "customer_first_name",
        "customer_last_name",
        "segment",
        "country",
        "execution_datetime", 
        "source_file" 
    )
    .agg(
        F.sum(F.when(F.col("is_last_1m"), 1).otherwise(0)).alias("orders_last_1_month"),
        F.sum(F.when(F.col("is_last_6m"), 1).otherwise(0)).alias("orders_last_6_months"),
        F.sum(F.when(F.col("is_last_12m"), 1).otherwise(0)).alias("orders_last_12_months"),
        F.count("*").alias("orders_all_time")
    )
)

log("Gold-Customer", "Aggregation completed successfully")

In [0]:
customer_df.display()

# Write to Gold Customer


In [0]:
log("Gold-Customer", "Writing gold.customer table")

row_count = customer_df.count()

if row_count == 0:
    raise ValueError("Customer dataset is empty — stopping pipeline")

try:
    customer_df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("mergeSchema", "true") \
        .saveAsTable("gold.customer")

    log("Gold-Customer", f"Write successful | rows={row_count}")

except Exception as e:
    log("Gold-Customer", f"ERROR during write: {str(e)}")
    raise

# Gold Sales


In [0]:
log("Gold-Sales", "Creating sales dataset")

required_sales_cols = ["order_id", "order_date", "ship_date", "ship_mode", "city", "execution_datetime", "source_file"]

missing_cols = [c for c in required_sales_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required sales columns: {missing_cols}")

sales_df = df.select(*required_sales_cols)

log("Gold-Sales", f"Sales rows: {sales_df.count()}")

In [0]:
sales_df.display()

In [0]:
log("Gold-Sales", "Writing gold.sales table")

try:
    sales_df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("mergeSchema", "true") \
        .saveAsTable("gold.sales")

    log("Gold-Sales", "Write successful")

except Exception as e:
    log("Gold-Sales", f"ERROR during write: {str(e)}")
    raise